# Augment HC3 With Llama 3

This notebook does four things:

1. Downloads `Hello-SimpleAI/HC3` if it is not already cached locally.
2. Checks whether CUDA is available.
3. Loads a Llama 3 generation model.
4. Shows the exact prompt sent to the model on one example, then generates a new answer column and adds it to the dataset.

Before running the model-loading cells on your HPC, make sure your environment has the project dependencies installed and your Hugging Face account has access to the chosen Llama checkpoint.

In [1]:
from pathlib import Path

requirements_path = Path.cwd().resolve() / "requirements.txt"
if not requirements_path.exists():
    raise FileNotFoundError(f"requirements.txt not found at {requirements_path}")

TORCH_VERSION = "2.4.1"
TORCH_INDEX_URL = "https://download.pytorch.org/whl/cu121"

print(f"Installing project dependencies from: {requirements_path}")
%pip install -r {requirements_path}

print("Reinstalling a V100-friendly PyTorch wheel from the official PyTorch CUDA 11.8 index...")
%pip install --upgrade --force-reinstall --index-url {TORCH_INDEX_URL} torch=={TORCH_VERSION}

print("If torch was changed in this cell, restart the kernel before running the next cell.")

Installing project dependencies from: /gpfs/helios/home/gauthierbernarda/git/nlp-ai-detection/requirements.txt
Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.
Reinstalling a V100-friendly PyTorch wheel from the official PyTorch CUDA 11.8 index...
Defaulting to user installation because normal site-packages is not writeable
Looking in indexes: https://download.pytorch.org/whl/cu121
  Using cached https://download-r2.pytorch.org/whl/cu121/torch-2.4.1%2Bcu121-cp312-cp312-linux_x86_64.whl (798.9 MB)
  Using cached filelock-3.25.2-py3-none-any.whl.metadata (2.0 kB)
  Using cached https://download.pytorch.org/whl/typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.6.1-py3-none-any.whl.metadata (6.8 kB)
  Using cached https://download.pytorch.org/whl/jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Usin

In [2]:
from pathlib import Path
import json
import sys

import torch
from IPython.display import Markdown, display

REPO_ROOT = Path.cwd().resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from scripts.augment_hc3 import (
    DEFAULT_DATASET_REVISION,
    augment_dataset_dict,
    build_generation_kwargs_from_values,
    detect_backend,
    dtype_to_name,
    ensure_local_dataset,
    generate_from_prompts,
    load_generation_pipeline,
    normalize_dataset_object,
    render_prompt_for_row,
)
from datasets import load_from_disk


/gpfs/space/software/cluster_software/environments/jupyterlab/python-3.12.3/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Configuration

Change these values before running the notebook. Set `PROMPT_TEMPLATE = None` if you want to generate answers with no extra conditioning.

Use `INPUT_DATASET_DIR = RAW_DATASET_DIR` together with `RANDOM_SUBSET_SIZE = 5000` to create and save a reusable random 5k subset on the first run. Later, point `INPUT_DATASET_DIR` at that saved subset directory and set `RANDOM_SUBSET_SIZE = None` to add new columns without resampling.

In [3]:
DATASET_NAME = "Hello-SimpleAI/HC3"
DATASET_CONFIG = "all"
DATASET_REVISION = DEFAULT_DATASET_REVISION
RAW_DATASET_DIR = Path("data/hc3_raw")
INPUT_DATASET_DIR = RAW_DATASET_DIR
AUGMENTED_DATASET_DIR = Path("data/hc3_llama3_subset5k")

MODEL_NAME = "meta-llama/Llama-3.2-3B-Instruct"
TRUST_REMOTE_CODE = False
REQUESTED_DEVICE = "auto"
TORCH_DTYPE = "auto"

QUESTION_COLUMN = "question"
SOURCE_COLUMN = "source"
NEW_COLUMN_NAME = "llama3.2-3B-instruct-answers"
PROMPT_COLUMN_NAME = None

PROMPT_TEMPLATE = None
SYSTEM_PROMPT = None
DISABLE_CHAT_TEMPLATE = False

TARGET_SPLITS = None
RANDOM_SUBSET_SIZE = 5000
MAX_SAMPLES_PER_SPLIT = None
OVERWRITE_COLUMN = False

BATCH_SIZE = 512
MAX_INPUT_LENGTH = 1024
MAX_NEW_TOKENS = 512
DO_SAMPLE = True
TEMPERATURE = 0.8
TOP_P = 0.95
NUM_BEAMS = 1
NUM_RETURN_SEQUENCES = 1

PREVIEW_SPLIT = "train"
PREVIEW_ROW_INDEX = 0
SEED = 42
SAVE_AUGMENTED_DATASET = True


## 1. Download HC3 Or Load It From Disk

In [4]:
if INPUT_DATASET_DIR.resolve() == RAW_DATASET_DIR.resolve():
    dataset_was_cached = RAW_DATASET_DIR.exists()
    dataset = ensure_local_dataset(
        RAW_DATASET_DIR,
        dataset_name=DATASET_NAME,
        dataset_config=DATASET_CONFIG,
        dataset_revision=DATASET_REVISION,
    )
    dataset_status = "Loaded from disk cache." if dataset_was_cached else "Downloaded from Hugging Face and cached locally."
else:
    if not INPUT_DATASET_DIR.exists():
        raise FileNotFoundError(f"Input dataset directory not found: {INPUT_DATASET_DIR}")
    dataset = normalize_dataset_object(load_from_disk(str(INPUT_DATASET_DIR)))
    dataset_status = "Loaded an existing dataset from disk."

print(f"Input dataset directory: {INPUT_DATASET_DIR}")
print(dataset_status)
for split_name, split_dataset in dataset.items():
    print(f"{split_name}: {len(split_dataset):,} rows")
print(f"Columns: {dataset[next(iter(dataset.keys()))].column_names}")


Input dataset directory: data/hc3_raw
Loaded from disk cache.
train: 24,322 rows
Columns: ['id', 'question', 'human_answers', 'chatgpt_answers', 'source']


## 2. Check CUDA

In [5]:
backend = detect_backend(REQUESTED_DEVICE)
device = torch.device(backend)

print(f"Requested device: {REQUESTED_DEVICE}")
print(f"Resolved backend: {backend}")
print(f"torch.cuda.is_available(): {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"CUDA device count: {torch.cuda.device_count()}")
    print(f"CUDA device name: {torch.cuda.get_device_name(0)}")
elif backend != "cuda":
    print("CUDA is not available. Generation will run on the resolved backend above.")


Requested device: auto
Resolved backend: cuda
torch.cuda.is_available(): True
CUDA device count: 1
CUDA device name: NVIDIA H200 NVL


## 3. Load The Llama 3 Model

In [6]:
model, tokenizer, is_encoder_decoder, resolved_torch_dtype = load_generation_pipeline(
    model_name=MODEL_NAME,
    backend=backend,
    trust_remote_code=TRUST_REMOTE_CODE,
    torch_dtype=TORCH_DTYPE,
)

print(f"Model: {MODEL_NAME}")
print(f"Encoder-decoder: {is_encoder_decoder}")
print(f"Model dtype: {dtype_to_name(resolved_torch_dtype)}")
print(f"Tokenizer has chat template: {bool(getattr(tokenizer, 'chat_template', None))}")


`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 254/254 [00:00<00:00, 1849.13it/s]


Model: meta-llama/Llama-3.2-3B-Instruct
Encoder-decoder: False
Model dtype: bfloat16
Tokenizer has chat template: True


## 4. Preview The Exact Prompt Sent To The Model

In [7]:
preview_row = dataset[PREVIEW_SPLIT][PREVIEW_ROW_INDEX]
preview = render_prompt_for_row(
    question=preview_row[QUESTION_COLUMN],
    tokenizer=tokenizer,
    source=preview_row.get(SOURCE_COLUMN, "") if isinstance(preview_row, dict) else "",
    split_name=PREVIEW_SPLIT,
    row_index=PREVIEW_ROW_INDEX,
    prompt_template=PROMPT_TEMPLATE,
    system_prompt_template=SYSTEM_PROMPT,
    disable_chat_template=DISABLE_CHAT_TEMPLATE,
)

display(Markdown("## Prompt Preview"))
display(Markdown(
    f"**Split:** `{PREVIEW_SPLIT}`  \n"
    f"**Row index:** `{PREVIEW_ROW_INDEX}`  \n"
    f"**Source:** `{preview['source'] or 'N/A'}`  \n"
    f"**Using chat template:** `{preview['use_chat_template']}`"
))
display(Markdown("### Question"))
display(Markdown(f"```text\n{preview['question']}\n```"))
display(Markdown("### User Prompt"))
display(Markdown(f"```text\n{preview['user_prompt']}\n```"))

if preview['system_prompt']:
    display(Markdown("### System Prompt"))
    display(Markdown(f"```text\n{preview['system_prompt']}\n```"))

display(Markdown("### Final Text Sent To The Model"))
display(Markdown(f"```text\n{preview['formatted_prompt']}\n```"))


## Prompt Preview

**Split:** `train`  
**Row index:** `0`  
**Source:** `reddit_eli5`  
**Using chat template:** `True`

### Question

```text
Why is every book I hear about a " NY Times # 1 Best Seller " ? ELI5 : Why is every book I hear about a " NY Times # 1 Best Seller " ? Should n't there only be one " # 1 " best seller ? Please explain like I'm five.
```

### User Prompt

```text
Why is every book I hear about a " NY Times # 1 Best Seller " ? ELI5 : Why is every book I hear about a " NY Times # 1 Best Seller " ? Should n't there only be one " # 1 " best seller ? Please explain like I'm five.
```

### Final Text Sent To The Model

```text
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 19 Apr 2026

<|eot_id|><|start_header_id|>user<|end_header_id|>

Why is every book I hear about a " NY Times # 1 Best Seller " ? ELI5 : Why is every book I hear about a " NY Times # 1 Best Seller " ? Should n't there only be one " # 1 " best seller ? Please explain like I'm five.<|eot_id|><|start_header_id|>assistant<|end_header_id|>


```

## 5. Generate One Example Answer

In [8]:
preview_generation_kwargs = build_generation_kwargs_from_values(
    tokenizer=tokenizer,
    batch_size=1,
    max_input_length=MAX_INPUT_LENGTH,
    max_new_tokens=MAX_NEW_TOKENS,
    do_sample=DO_SAMPLE,
    temperature=TEMPERATURE,
    top_p=TOP_P,
    num_beams=NUM_BEAMS,
    num_return_sequences=NUM_RETURN_SEQUENCES,
)

preview_answers = generate_from_prompts(
    [preview['formatted_prompt']],
    tokenizer=tokenizer,
    model=model,
    is_encoder_decoder=is_encoder_decoder,
    device=device,
    max_input_length=MAX_INPUT_LENGTH,
    generation_kwargs=preview_generation_kwargs,
)

display(Markdown("## Example Model Output"))
display(Markdown(f"```text\n{preview_answers[0][0]}\n```"))


## Example Model Output

```text
Imagine you have a lemonade stand. You make the best lemonade in your neighborhood, and everyone wants to buy it. But, there are many other kids in the neighborhood who also make delicious lemonade.

To figure out who makes the best lemonade, the whole neighborhood decides to have a contest. They count how many cups of lemonade each kid sells, and the kid with the most cups sells the best lemonade.

But, what if there are two kids who sell the same amount of lemonade? Shouldn't only one of them be the best? 

That's kind of like what's happening with the "NY Times #1 Best Seller" list. The New York Times (NYT) is like the neighborhood, and it's counting how many copies of each book are sold. They make a list of the top 10 (or sometimes more) books that are selling the most.

The problem is, sometimes there can be a tie, and the NYT can't just pick one book to be the "#1 Best Seller" because there are two or more books that are selling the same amount. So, they make a list of the top 10, and the books are ranked #1 to #10.

It's not like they're saying that two books are equally good or something, it's just that they're both selling the same amount of copies. The NYT is trying to give a list of the most popular books, and sometimes there are multiple books that fit that criteria.

It's like if you had a contest and there were two kids who tied for first place, you could say that they both won, or you could say that they tied and both won. Either way, it's still a win!

So, that's why you hear about multiple "NY Times #1 Best Sellers" – it's just a way to show that there are several popular books, and no one book is necessarily the absolute best, but they're all very popular!
```

## 6. Run Dataset Augmentation

Set `RANDOM_SUBSET_SIZE = 5000` when you want to freeze and save a reusable random subset from the raw HC3 dataset.

On later runs, point `INPUT_DATASET_DIR` at the saved subset and set `RANDOM_SUBSET_SIZE = None` so you add new columns on top of the exact same rows without resampling.

For a quick smoke test, you can still use `MAX_SAMPLES_PER_SPLIT`.

In [9]:
augmented_dataset = augment_dataset_dict(
    dataset,
    tokenizer=tokenizer,
    model=model,
    is_encoder_decoder=is_encoder_decoder,
    device=device,
    column_name=NEW_COLUMN_NAME,
    prompt_column_name=PROMPT_COLUMN_NAME,
    question_column=QUESTION_COLUMN,
    source_column=SOURCE_COLUMN,
    splits=TARGET_SPLITS,
    seed=SEED,
    random_subset_size=RANDOM_SUBSET_SIZE,
    max_samples_per_split=MAX_SAMPLES_PER_SPLIT,
    prompt_template=PROMPT_TEMPLATE,
    system_prompt_template=SYSTEM_PROMPT,
    disable_chat_template=DISABLE_CHAT_TEMPLATE,
    overwrite_column=OVERWRITE_COLUMN,
    batch_size=BATCH_SIZE,
    max_input_length=MAX_INPUT_LENGTH,
    max_new_tokens=MAX_NEW_TOKENS,
    do_sample=DO_SAMPLE,
    temperature=TEMPERATURE,
    top_p=TOP_P,
    num_beams=NUM_BEAMS,
    num_return_sequences=NUM_RETURN_SEQUENCES,
)

print(f"Added column: {NEW_COLUMN_NAME}")
if RANDOM_SUBSET_SIZE is not None:
    print(f"Random subset size requested: {RANDOM_SUBSET_SIZE}")
print(f"Preview generated answers: {augmented_dataset[PREVIEW_SPLIT][PREVIEW_ROW_INDEX][NEW_COLUMN_NAME]}")
if PROMPT_COLUMN_NAME is not None:
    print(f"Stored prompt column: {PROMPT_COLUMN_NAME}")


[train] Generated 512/5,000 rows
[train] Generated 1,024/5,000 rows
[train] Generated 1,536/5,000 rows
[train] Generated 2,048/5,000 rows
[train] Generated 2,560/5,000 rows
[train] Generated 3,072/5,000 rows
[train] Generated 3,584/5,000 rows
[train] Generated 4,096/5,000 rows
[train] Generated 4,608/5,000 rows
[train] Generated 5,000/5,000 rows


Flattening the indices: 100%|██████████| 5000/5000 [00:00<00:00, 71043.18 examples/s]

Added column: llama3.2-3B-instruct-answers
Random subset size requested: 5000
Preview generated answers: ['Let\'s talk about eyes!\n\nSo, you know how people have different colored eyes, like blue, brown, green, and even gray? Well, it\'s all because of the way our eyes are made.\n\nOur eyes are made up of tiny things called melanin, which is like a special kind of pigment. Melanin helps protect our eyes from the sun\'s rays and gives our eyes their color.\n\nThe amount and kind of melanin in our eyes determines the color of our eyes. Here\'s why:\n\n1. **Brown eyes**: People with brown eyes have a lot of melanin in their eyes. It\'s like they have a special sunblock for their eyes!\n2. **Blue eyes**: People with blue eyes have less melanin, so their eyes look more transparent. It\'s like they have a special window to see the world!\n3. **Green eyes**: People with green eyes have a mix of melanin and a special helper called lipochrome. It\'s like a secret ingredient that makes their ey

## 7. Save The Augmented Dataset

In [10]:
run_config = {
    'dataset_name': DATASET_NAME,
    'dataset_config': DATASET_CONFIG,
    'dataset_revision': DATASET_REVISION,
    'raw_dataset_dir': str(RAW_DATASET_DIR),
    'input_dataset_dir': str(INPUT_DATASET_DIR),
    'augmented_dataset_dir': str(AUGMENTED_DATASET_DIR),
    'model_name': MODEL_NAME,
    'backend': backend,
    'torch_dtype': dtype_to_name(resolved_torch_dtype),
    'question_column': QUESTION_COLUMN,
    'source_column': SOURCE_COLUMN,
    'new_column_name': NEW_COLUMN_NAME,
    'prompt_column_name': PROMPT_COLUMN_NAME,
    'prompt_template': PROMPT_TEMPLATE,
    'system_prompt': SYSTEM_PROMPT,
    'disable_chat_template': DISABLE_CHAT_TEMPLATE,
    'target_splits': TARGET_SPLITS,
    'seed': SEED,
    'random_subset_size': RANDOM_SUBSET_SIZE,
    'max_samples_per_split': MAX_SAMPLES_PER_SPLIT,
    'batch_size': BATCH_SIZE,
    'max_input_length': MAX_INPUT_LENGTH,
    'max_new_tokens': MAX_NEW_TOKENS,
    'do_sample': DO_SAMPLE,
    'temperature': TEMPERATURE,
    'top_p': TOP_P,
    'num_beams': NUM_BEAMS,
    'num_return_sequences': NUM_RETURN_SEQUENCES,
}

if SAVE_AUGMENTED_DATASET:
    AUGMENTED_DATASET_DIR.parent.mkdir(parents=True, exist_ok=True)
    augmented_dataset.save_to_disk(str(AUGMENTED_DATASET_DIR))
    with (AUGMENTED_DATASET_DIR / 'augmentation_run_config.json').open('w', encoding='utf-8') as handle:
        json.dump(run_config, handle, indent=2, ensure_ascii=False)
    print(f"Saved augmented dataset to: {AUGMENTED_DATASET_DIR}")
else:
    print("SAVE_AUGMENTED_DATASET is False, so nothing was written to disk.")


Saving the dataset (1/1 shards): 100%|██████████| 5000/5000 [00:00<00:00, 388598.96 examples/s]

Saved augmented dataset to: data/hc3_llama3_subset5k
